In [1]:
import pandas as pd
import numpy as np

# 读入全球疫苗数据
df = pd.read_csv("cleaned_vaccinations_global.csv")

# 日期转格式
df["date"] = pd.to_datetime(df["date"])

# 只保留分析需要的字段（如果有缺字段会报错，方便你及时发现）
needed_cols = [
    "country",
    "date",
    "daily_vaccinations_smoothed",
    "daily_people_vaccinated_smoothed",
    "daily_people_vaccinated_smoothed_per_hundred",
    "people_vaccinated_interpolated",
    "people_fully_vaccinated_interpolated",
    "total_boosters_interpolated"
]

df = df[needed_cols].copy()

# 按国家+日期排序
df = df.sort_values(["country", "date"]).reset_index(drop=True)

df.head()

,country,date,daily_vaccinations_smoothed,daily_people_vaccinated_smoothed,daily_people_vaccinated_smoothed_per_hundred,people_vaccinated_interpolated,people_fully_vaccinated_interpolated,total_boosters_interpolated
0,Afghanistan,2021-02-23,1366.666667,1366.666667,0.003368,1366.666667,55624.0,0.0
1,Afghanistan,2021-02-24,1366.666667,1366.666667,0.003368,2733.333333,55624.0,0.0
2,Afghanistan,2021-02-25,1366.666667,1366.666667,0.003368,4100.000000,55624.0,0.0
3,Afghanistan,2021-02-26,1366.666667,1366.666667,0.003368,5466.666667,55624.0,0.0
4,Afghanistan,2021-02-27,1366.666667,1366.666667,0.003368,6833.333333,55624.0,0.0


In [2]:
quality_rows = []

for country, temp in df.groupby("country"):
    temp = temp.sort_values("date").copy()

    # 最终累计值
    final_people_vaccinated = temp["people_vaccinated_interpolated"].max()
    final_people_fully_vaccinated = temp["people_fully_vaccinated_interpolated"].max()
    final_boosters = temp["total_boosters_interpolated"].max()

    # 累计接种变化
    temp["vacc_diff"] = temp["people_vaccinated_interpolated"].diff()

    # 第一次累计 > 0
    first_nonzero_date = temp.loc[
        temp["people_vaccinated_interpolated"] > 0, "date"
    ].min()

    # 第一次发生变化
    first_change_idx = temp.index[temp["vacc_diff"] > 0]
    if len(first_change_idx) > 0:
        first_change_idx = first_change_idx[0]
        first_change_date = temp.loc[first_change_idx, "date"]
        baseline_at_first_change = temp.loc[first_change_idx, "people_vaccinated_interpolated"]
    else:
        first_change_date = pd.NaT
        baseline_at_first_change = np.nan

    # 起点占最终值比例
    if pd.notna(final_people_vaccinated) and final_people_vaccinated > 0 and pd.notna(baseline_at_first_change):
        baseline_ratio = baseline_at_first_change / final_people_vaccinated
    else:
        baseline_ratio = np.nan

    # 非零覆盖比例（看是不是长期都是0）
    vacc_nonzero_ratio = (temp["people_vaccinated_interpolated"] > 0).mean()
    full_nonzero_ratio = (temp["people_fully_vaccinated_interpolated"] > 0).mean()
    booster_nonzero_ratio = (temp["total_boosters_interpolated"] > 0).mean()

    # 数据跨度
    min_date = temp["date"].min()
    max_date = temp["date"].max()
    n_days = temp["date"].nunique()

    quality_rows.append({
        "country": country,
        "min_date": min_date,
        "max_date": max_date,
        "n_days": n_days,
        "first_nonzero_date": first_nonzero_date,
        "first_change_date": first_change_date,
        "baseline_at_first_change": baseline_at_first_change,
        "final_people_vaccinated": final_people_vaccinated,
        "final_people_fully_vaccinated": final_people_fully_vaccinated,
        "final_boosters": final_boosters,
        "baseline_ratio": baseline_ratio,
        "vacc_nonzero_ratio": vacc_nonzero_ratio,
        "full_nonzero_ratio": full_nonzero_ratio,
        "booster_nonzero_ratio": booster_nonzero_ratio
    })

country_quality_table = pd.DataFrame(quality_rows)
country_quality_table.head()

,country,min_date,max_date,n_days,first_nonzero_date,first_change_date,baseline_at_first_change,final_people_vaccinated,final_people_fully_vaccinated,final_boosters,baseline_ratio,vacc_nonzero_ratio,full_nonzero_ratio,booster_nonzero_ratio
0,Afghanistan,2021-02-23,2023-12-31,1042,2021-02-23,2021-02-24,2.733333e+03,19151369.0,18370386.0,2729940.0,0.000143,1.0,1.0,0.589251
1,Africa,2020-12-02,2024-08-12,1350,2020-12-02,2021-01-10,3.146276e+06,557793381.0,463324309.0,100592192.0,0.005641,1.0,1.0,1.000000
2,Albania,2021-01-11,2023-09-10,973,2021-01-11,2021-01-12,1.280000e+02,1349255.0,1279333.0,402371.0,0.000095,1.0,1.0,1.000000
3,Algeria,2021-01-30,2022-09-04,583,2021-01-30,2021-01-31,3.778500e+03,7840131.0,6481186.0,575651.0,0.000482,1.0,1.0,1.000000
4,Andorra,2021-01-26,2023-09-24,972,2021-01-26,2021-01-27,7.074286e+02,57913.0,53501.0,43071.0,0.012215,1.0,1.0,1.000000


In [3]:
def classify_rollout_quality(row):
    if pd.isna(row["baseline_ratio"]):
        return "bad_no_rollout_signal"
    elif row["baseline_ratio"] <= 0.02:
        return "good_for_rollout"
    elif row["baseline_ratio"] <= 0.10:
        return "usable_with_caution"
    else:
        return "late_start_recording"

country_quality_table["rollout_quality"] = country_quality_table.apply(classify_rollout_quality, axis=1)

# 是否适合 rollout 起步分析
country_quality_table["use_for_rollout_start_analysis"] = country_quality_table["rollout_quality"].isin([
    "good_for_rollout", "usable_with_caution"
])

# 是否适合覆盖率/中后期分析
country_quality_table["use_for_coverage_analysis"] = (
    country_quality_table["final_people_vaccinated"] > 0
)

country_quality_table.sort_values("baseline_ratio", ascending=False).head(20)

,country,min_date,max_date,n_days,first_nonzero_date,first_change_date,baseline_at_first_change,final_people_vaccinated,final_people_fully_vaccinated,final_boosters,baseline_ratio,vacc_nonzero_ratio,full_nonzero_ratio,booster_nonzero_ratio,rollout_quality,use_for_rollout_start_analysis,use_for_coverage_analysis
178,Saudi Arabia,2021-01-07,2023-04-25,839,2021-01-07,2021-06-28,1.597756e+07,2.704136e+07,2.543394e+07,1.079398e+07,0.590856,1.0,1.000000,1.000000,late_start_recording,False,True
30,British Virgin Islands,2021-05-22,2022-09-16,483,2021-05-22,2021-05-23,1.134014e+04,1.946600e+04,1.826100e+04,3.726000e+03,0.582562,1.0,1.000000,1.000000,late_start_recording,False,True
184,Sint Maarten (Dutch part),2021-05-08,2023-03-31,693,2021-05-08,2021-05-09,1.485686e+04,2.978800e+04,2.677300e+04,9.146000e+03,0.498753,1.0,1.000000,1.000000,late_start_recording,False,True
106,Jersey,2021-03-15,2023-01-29,686,2021-03-15,2021-03-16,4.099386e+04,8.436500e+04,8.188200e+04,7.366200e+04,0.485911,1.0,1.000000,1.000000,late_start_recording,False,True
43,China,2020-12-16,2023-04-20,856,2020-12-16,2021-06-11,6.278506e+08,1.318027e+09,1.284480e+09,8.340601e+08,0.476357,1.0,1.000000,1.000000,late_start_recording,False,True
100,Ireland,2021-05-21,2024-05-10,1086,2021-05-21,2021-05-22,1.875421e+06,4.112237e+06,4.065584e+06,3.211763e+06,0.456059,1.0,1.000000,0.882136,late_start_recording,False,True
47,Cook Islands,2021-05-26,2023-02-13,629,2021-05-26,2021-05-27,5.485286e+03,1.511200e+04,1.472800e+04,1.024000e+04,0.362975,1.0,0.988871,1.000000,late_start_recording,False,True
26,Bonaire Sint Eustatius and Saba,2021-04-10,2021-09-01,145,2021-04-10,2021-04-11,5.910593e+03,1.910900e+04,1.673600e+04,0.000000e+00,0.309309,1.0,1.000000,0.000000,late_start_recording,False,True
136,Monaco,2020-12-31,2023-12-23,1088,2020-12-31,2021-03-05,8.535733e+03,2.887500e+04,2.566700e+04,1.774700e+04,0.295610,1.0,1.000000,1.000000,late_start_recording,False,True
218,Upper-middle-income countries,2020-12-02,2024-08-12,1350,2020-12-02,2020-12-25,6.259912e+08,2.318506e+09,2.172816e+09,1.360163e+09,0.269998,1.0,1.000000,1.000000,late_start_recording,False,True


In [4]:
problem_countries = country_quality_table[
    country_quality_table["rollout_quality"] == "late_start_recording"
].sort_values("baseline_ratio", ascending=False)

problem_countries[[
    "country",
    "first_nonzero_date",
    "first_change_date",
    "baseline_at_first_change",
    "final_people_vaccinated",
    "baseline_ratio"
]].head(30)

,country,first_nonzero_date,first_change_date,baseline_at_first_change,final_people_vaccinated,baseline_ratio
178,Saudi Arabia,2021-01-07,2021-06-28,1.597756e+07,2.704136e+07,0.590856
30,British Virgin Islands,2021-05-22,2021-05-23,1.134014e+04,1.946600e+04,0.582562
184,Sint Maarten (Dutch part),2021-05-08,2021-05-09,1.485686e+04,2.978800e+04,0.498753
106,Jersey,2021-03-15,2021-03-16,4.099386e+04,8.436500e+04,0.485911
43,China,2020-12-16,2021-06-11,6.278506e+08,1.318027e+09,0.476357
100,Ireland,2021-05-21,2021-05-22,1.875421e+06,4.112237e+06,0.456059
47,Cook Islands,2021-05-26,2021-05-27,5.485286e+03,1.511200e+04,0.362975
26,Bonaire Sint Eustatius and Saba,2021-04-10,2021-04-11,5.910593e+03,1.910900e+04,0.309309
136,Monaco,2020-12-31,2021-03-05,8.535733e+03,2.887500e+04,0.295610
218,Upper-middle-income countries,2020-12-02,2020-12-25,6.259912e+08,2.318506e+09,0.269998


In [5]:
rollout_ok_countries = country_quality_table.loc[
    country_quality_table["use_for_rollout_start_analysis"], "country"
].tolist()

df_rollout_clean = df[df["country"].isin(rollout_ok_countries)].copy()

print("用于 rollout 起步分析的国家数：", df_rollout_clean["country"].nunique())

用于 rollout 起步分析的国家数： 208


In [6]:
coverage_ok_countries = country_quality_table.loc[
    country_quality_table["use_for_coverage_analysis"], "country"
].tolist()

df_coverage_clean = df[df["country"].isin(coverage_ok_countries)].copy()

print("用于 coverage / heatmap 分析的国家数：", df_coverage_clean["country"].nunique())

用于 coverage / heatmap 分析的国家数： 229


In [7]:
aligned_rows = []

for country, temp in df_rollout_clean.groupby("country"):
    temp = temp.sort_values("date").copy()
    temp["vacc_diff"] = temp["people_vaccinated_interpolated"].diff()

    first_change_date = temp.loc[temp["vacc_diff"] > 0, "date"].min()

    if pd.isna(first_change_date):
        continue

    temp = temp[temp["date"] >= first_change_date].copy()
    temp["rollout_day"] = (temp["date"] - first_change_date).dt.days

    aligned_rows.append(temp)

aligned_rollout_df = pd.concat(aligned_rows, ignore_index=True)
aligned_rollout_df.head()

,country,date,daily_vaccinations_smoothed,daily_people_vaccinated_smoothed,daily_people_vaccinated_smoothed_per_hundred,people_vaccinated_interpolated,people_fully_vaccinated_interpolated,total_boosters_interpolated,vacc_diff,rollout_day
0,Afghanistan,2021-02-24,1366.666667,1366.666667,0.003368,2733.333333,55624.0,0.0,1366.666667,0
1,Afghanistan,2021-02-25,1366.666667,1366.666667,0.003368,4100.000000,55624.0,0.0,1366.666667,1
2,Afghanistan,2021-02-26,1366.666667,1366.666667,0.003368,5466.666667,55624.0,0.0,1366.666667,2
3,Afghanistan,2021-02-27,1366.666667,1366.666667,0.003368,6833.333333,55624.0,0.0,1366.666667,3
4,Afghanistan,2021-02-28,1366.666667,1366.666667,0.003368,8200.000000,55624.0,0.0,1366.666667,4


In [8]:
df_coverage_clean["quarter"] = df_coverage_clean["date"].dt.to_period("Q").astype(str)
df_coverage_clean.head()

,country,date,daily_vaccinations_smoothed,daily_people_vaccinated_smoothed,daily_people_vaccinated_smoothed_per_hundred,people_vaccinated_interpolated,people_fully_vaccinated_interpolated,total_boosters_interpolated,quarter
0,Afghanistan,2021-02-23,1366.666667,1366.666667,0.003368,1366.666667,55624.0,0.0,2021Q1
1,Afghanistan,2021-02-24,1366.666667,1366.666667,0.003368,2733.333333,55624.0,0.0,2021Q1
2,Afghanistan,2021-02-25,1366.666667,1366.666667,0.003368,4100.000000,55624.0,0.0,2021Q1
3,Afghanistan,2021-02-26,1366.666667,1366.666667,0.003368,5466.666667,55624.0,0.0,2021Q1
4,Afghanistan,2021-02-27,1366.666667,1366.666667,0.003368,6833.333333,55624.0,0.0,2021Q1


In [9]:
country_quality_table.to_csv("country_quality_table.csv", index=False)
df_rollout_clean.to_csv("df_rollout_clean.csv", index=False)
aligned_rollout_df.to_csv("aligned_rollout_df.csv", index=False)
df_coverage_clean.to_csv("df_coverage_clean.csv", index=False)

In [11]:
country_quality_table.to_csv("country_quality_table.csv", index=False)
problem_countries.head(30).to_csv("problem_countries_top30.csv", index=False)

country_quality_table.sort_values("baseline_ratio", ascending=False).head(20).to_csv(
    "baseline_ratio_highest_20.csv", index=False
)

country_quality_table.sort_values("baseline_ratio", ascending=True).head(20).to_csv(
    "baseline_ratio_lowest_20.csv", index=False
)

In [13]:
aggregate_groups = [
    "World",
    "Africa",
    "Asia",
    "Europe",
    "North America",
    "South America",
    "Oceania",
    "European Union (27)",
    "High-income countries",
    "Upper-middle-income countries",
    "Lower-middle-income countries",
    "Low-income countries"
]

In [14]:
aggregate_quality = country_quality_table[
    country_quality_table["country"].isin(aggregate_groups)
].copy()

aggregate_quality[[
    "country",
    "first_nonzero_date",
    "first_change_date",
    "baseline_ratio",
    "rollout_quality",
    "use_for_rollout_start_analysis",
    "use_for_coverage_analysis"
]].sort_values("country")

,country,first_nonzero_date,first_change_date,baseline_ratio,rollout_quality,use_for_rollout_start_analysis,use_for_coverage_analysis
1,Africa,2020-12-02,2021-01-10,0.005641,good_for_rollout,True,True
11,Asia,2020-12-02,2020-12-20,0.173690,late_start_recording,False,True
68,Europe,2020-12-02,2020-12-03,0.008792,good_for_rollout,True,True
69,European Union (27),2020-12-02,2020-12-05,0.006542,good_for_rollout,True,True
91,High-income countries,2020-12-02,2020-12-03,0.019847,good_for_rollout,True,True
122,Low-income countries,2020-12-02,2021-02-16,0.002825,good_for_rollout,True,True
123,Lower-middle-income countries,2020-12-02,2021-01-13,0.001453,good_for_rollout,True,True
153,North America,2020-12-02,2020-12-14,0.000428,good_for_rollout,True,True
156,Oceania,2020-12-02,2021-02-03,0.000734,good_for_rollout,True,True
190,South America,2020-12-02,2020-12-25,0.002080,good_for_rollout,True,True


In [15]:
df_group_coverage = df_coverage_clean[
    df_coverage_clean["country"].isin(aggregate_groups)
].copy()

df_group_coverage["quarter"] = df_group_coverage["date"].dt.to_period("Q").astype(str)

print(df_group_coverage["country"].unique())
df_group_coverage.head()

['Africa' 'Asia' 'Europe' 'European Union (27)' 'High-income countries'
 'Low-income countries' 'Lower-middle-income countries' 'North America'
 'Oceania' 'South America' 'Upper-middle-income countries' 'World']


,country,date,daily_vaccinations_smoothed,daily_people_vaccinated_smoothed,daily_people_vaccinated_smoothed_per_hundred,people_vaccinated_interpolated,people_fully_vaccinated_interpolated,total_boosters_interpolated,quarter
1042,Africa,2020-12-02,0.0,0.0,0.0,3145776.0,19629346.0,8763139.0,2020Q4
1043,Africa,2020-12-03,0.0,0.0,0.0,3145776.0,19629346.0,8763139.0,2020Q4
1044,Africa,2020-12-04,0.0,0.0,0.0,3145776.0,19629346.0,8763139.0,2020Q4
1045,Africa,2020-12-05,0.0,0.0,0.0,3145776.0,19629346.0,8763139.0,2020Q4
1046,Africa,2020-12-06,0.0,0.0,0.0,3145776.0,19629346.0,8763139.0,2020Q4


In [16]:
group_quarterly = (
    df_group_coverage.groupby(["country", "quarter"])
    .agg(
        rollout_speed=("daily_people_vaccinated_smoothed_per_hundred", "mean"),
        people_vaccinated=("people_vaccinated_interpolated", "mean"),
        people_fully_vaccinated=("people_fully_vaccinated_interpolated", "mean"),
        boosters=("total_boosters_interpolated", "mean")
    )
    .reset_index()
)

group_quarterly.head(20)

,country,quarter,rollout_speed,people_vaccinated,people_fully_vaccinated,boosters
0,Africa,2020Q4,0.000000,3.145776e+06,1.962935e+07,8.763139e+06
1,Africa,2021Q1,0.004613,4.650873e+06,2.034902e+07,8.763139e+06
2,Africa,2021Q2,0.020179,2.210211e+07,2.741761e+07,8.763139e+06
3,Africa,2021Q3,0.044275,6.105430e+07,5.294370e+07,8.763139e+06
4,Africa,2021Q4,0.077383,1.441697e+08,1.115060e+08,9.082849e+06
5,Africa,2022Q1,0.065326,2.423278e+08,1.799265e+08,1.299179e+07
6,Africa,2022Q2,0.042637,3.086731e+08,2.349329e+08,2.400472e+07
7,Africa,2022Q3,0.055789,3.824896e+08,3.018044e+08,4.586434e+07
8,Africa,2022Q4,0.055936,4.553812e+08,3.684664e+08,6.097943e+07
9,Africa,2023Q1,0.025177,5.023765e+08,4.151825e+08,7.654225e+07


In [17]:
group_final_values = (
    df_group_coverage.groupby("country")
    .agg(
        final_people_vaccinated=("people_vaccinated_interpolated", "max"),
        final_people_fully_vaccinated=("people_fully_vaccinated_interpolated", "max")
    )
    .reset_index()
)

df_group_coverage = df_group_coverage.merge(group_final_values, on="country", how="left")

df_group_coverage["vaccinated_progress_ratio"] = (
    df_group_coverage["people_vaccinated_interpolated"] / df_group_coverage["final_people_vaccinated"]
)

df_group_coverage["fully_vaccinated_progress_ratio"] = (
    df_group_coverage["people_fully_vaccinated_interpolated"] / df_group_coverage["final_people_fully_vaccinated"]
)

df_group_coverage.head()

,country,date,daily_vaccinations_smoothed,daily_people_vaccinated_smoothed,daily_people_vaccinated_smoothed_per_hundred,people_vaccinated_interpolated,people_fully_vaccinated_interpolated,total_boosters_interpolated,quarter,final_people_vaccinated,final_people_fully_vaccinated,vaccinated_progress_ratio,fully_vaccinated_progress_ratio
0,Africa,2020-12-02,0.0,0.0,0.0,3145776.0,19629346.0,8763139.0,2020Q4,557793381.0,463324309.0,0.00564,0.042366
1,Africa,2020-12-03,0.0,0.0,0.0,3145776.0,19629346.0,8763139.0,2020Q4,557793381.0,463324309.0,0.00564,0.042366
2,Africa,2020-12-04,0.0,0.0,0.0,3145776.0,19629346.0,8763139.0,2020Q4,557793381.0,463324309.0,0.00564,0.042366
3,Africa,2020-12-05,0.0,0.0,0.0,3145776.0,19629346.0,8763139.0,2020Q4,557793381.0,463324309.0,0.00564,0.042366
4,Africa,2020-12-06,0.0,0.0,0.0,3145776.0,19629346.0,8763139.0,2020Q4,557793381.0,463324309.0,0.00564,0.042366


In [18]:
group_quarterly = (
    df_group_coverage.groupby(["country", "quarter"])
    .agg(
        rollout_speed=("daily_people_vaccinated_smoothed_per_hundred", "mean"),
        first_dose_progress=("vaccinated_progress_ratio", "mean"),
        full_vaccination_progress=("fully_vaccinated_progress_ratio", "mean")
    )
    .reset_index()
)

group_quarterly.head(20)

,country,quarter,rollout_speed,first_dose_progress,full_vaccination_progress
0,Africa,2020Q4,0.000000,0.005640,0.042366
1,Africa,2021Q1,0.004613,0.008338,0.043920
2,Africa,2021Q2,0.020179,0.039624,0.059176
3,Africa,2021Q3,0.044275,0.109457,0.114269
4,Africa,2021Q4,0.077383,0.258464,0.240665
5,Africa,2022Q1,0.065326,0.434440,0.388338
6,Africa,2022Q2,0.042637,0.553382,0.507059
7,Africa,2022Q3,0.055789,0.685719,0.651389
8,Africa,2022Q4,0.055936,0.816398,0.795267
9,Africa,2023Q1,0.025177,0.900650,0.896095


In [19]:
aggregate_quality.to_csv("aggregate_quality_table.csv", index=False)
group_quarterly.to_csv("group_quarterly_rollout.csv", index=False)

In [20]:
group_meta = pd.DataFrame([
    {"country": "Africa", "group_type": "region", "display_order": 1},
    {"country": "Asia", "group_type": "region", "display_order": 2},
    {"country": "Europe", "group_type": "region", "display_order": 3},
    {"country": "North America", "group_type": "region", "display_order": 4},
    {"country": "South America", "group_type": "region", "display_order": 5},
    {"country": "Oceania", "group_type": "region", "display_order": 6},
    {"country": "European Union (27)", "group_type": "region_special", "display_order": 7},

    {"country": "High-income countries", "group_type": "income", "display_order": 8},
    {"country": "Upper-middle-income countries", "group_type": "income", "display_order": 9},
    {"country": "Lower-middle-income countries", "group_type": "income", "display_order": 10},
    {"country": "Low-income countries", "group_type": "income", "display_order": 11},

    {"country": "World", "group_type": "global", "display_order": 12},
])

group_meta

,country,group_type,display_order
0,Africa,region,1
1,Asia,region,2
2,Europe,region,3
3,North America,region,4
4,South America,region,5
5,Oceania,region,6
6,European Union (27),region_special,7
7,High-income countries,income,8
8,Upper-middle-income countries,income,9
9,Lower-middle-income countries,income,10


In [21]:
group_quarterly = pd.read_csv("group_quarterly_rollout.csv")
group_quarterly = group_quarterly.merge(group_meta, on="country", how="left")

group_quarterly.head()

,country,quarter,rollout_speed,first_dose_progress,full_vaccination_progress,group_type,display_order
0,Africa,2020Q4,0.000000,0.005640,0.042366,region,1
1,Africa,2021Q1,0.004613,0.008338,0.043920,region,1
2,Africa,2021Q2,0.020179,0.039624,0.059176,region,1
3,Africa,2021Q3,0.044275,0.109457,0.114269,region,1
4,Africa,2021Q4,0.077383,0.258464,0.240665,region,1


In [22]:
group_meta.to_csv("group_meta.csv", index=False)
group_quarterly.to_csv("group_quarterly_rollout_enriched.csv", index=False)

In [23]:
import pandas as pd
import numpy as np

group_quarterly = pd.read_csv("group_quarterly_rollout.csv")
group_meta = pd.read_csv("group_meta.csv")

group_quarterly = group_quarterly.merge(group_meta, on="country", how="left")
group_quarterly.head()

,country,quarter,rollout_speed,first_dose_progress,full_vaccination_progress,group_type,display_order
0,Africa,2020Q4,0.000000,0.005640,0.042366,region,1
1,Africa,2021Q1,0.004613,0.008338,0.043920,region,1
2,Africa,2021Q2,0.020179,0.039624,0.059176,region,1
3,Africa,2021Q3,0.044275,0.109457,0.114269,region,1
4,Africa,2021Q4,0.077383,0.258464,0.240665,region,1


In [24]:
group_quarterly_long = group_quarterly.melt(
    id_vars=["country", "quarter", "group_type", "display_order"],
    value_vars=["rollout_speed", "first_dose_progress", "full_vaccination_progress"],
    var_name="metric",
    value_name="value"
)

group_quarterly_long = group_quarterly_long.sort_values(
    ["group_type", "display_order", "quarter", "metric"]
).reset_index(drop=True)

group_quarterly_long.head(20)

,country,quarter,group_type,display_order,metric,value
0,World,2020Q4,global,12,first_dose_progress,0.115678
1,World,2020Q4,global,12,full_vaccination_progress,0.155793
2,World,2020Q4,global,12,rollout_speed,0.002326
3,World,2021Q1,global,12,first_dose_progress,0.138191
4,World,2021Q1,global,12,full_vaccination_progress,0.164560
5,World,2021Q1,global,12,rollout_speed,0.045423
6,World,2021Q2,global,12,first_dose_progress,0.250550
7,World,2021Q2,global,12,full_vaccination_progress,0.227545
8,World,2021Q2,global,12,rollout_speed,0.131465
9,World,2021Q3,global,12,first_dose_progress,0.503576


In [25]:
group_quarterly_long.to_csv("group_quarterly_long.csv", index=False)

In [26]:
ranking_rows = []

for metric in ["rollout_speed", "first_dose_progress", "full_vaccination_progress"]:
    temp = group_quarterly[["country", "quarter", "group_type", "display_order", metric]].copy()
    temp = temp.rename(columns={metric: "value"})
    
    temp["rank_within_quarter"] = temp.groupby("quarter")["value"].rank(
        method="min", ascending=False
    )
    temp["metric"] = metric
    
    ranking_rows.append(temp)

group_rankings_by_quarter = pd.concat(ranking_rows, ignore_index=True)

group_rankings_by_quarter = group_rankings_by_quarter.sort_values(
    ["metric", "quarter", "rank_within_quarter", "display_order"]
).reset_index(drop=True)

group_rankings_by_quarter.head(30)

,country,quarter,group_type,display_order,value,rank_within_quarter,metric
0,Upper-middle-income countries,2020Q4,income,9,0.269999,1.0,first_dose_progress
1,Asia,2020Q4,region,2,0.173729,2.0,first_dose_progress
2,World,2020Q4,global,12,0.115678,3.0,first_dose_progress
3,High-income countries,2020Q4,income,8,0.021344,4.0,first_dose_progress
4,Europe,2020Q4,region,3,0.009050,5.0,first_dose_progress
5,European Union (27),2020Q4,region_special,7,0.006607,6.0,first_dose_progress
6,Africa,2020Q4,region,1,0.005640,7.0,first_dose_progress
7,North America,2020Q4,region,4,0.003370,8.0,first_dose_progress
8,Low-income countries,2020Q4,income,11,0.002806,9.0,first_dose_progress
9,South America,2020Q4,region,5,0.002076,10.0,first_dose_progress


In [27]:
group_rankings_by_quarter.to_csv("group_rankings_by_quarter.csv", index=False)

In [28]:
group_qoq_change = group_quarterly.sort_values(["country", "quarter"]).copy()

for metric in ["rollout_speed", "first_dose_progress", "full_vaccination_progress"]:
    group_qoq_change[f"{metric}_qoq_change"] = group_qoq_change.groupby("country")[metric].diff()

group_qoq_change.head(20)

,country,quarter,rollout_speed,first_dose_progress,full_vaccination_progress,group_type,display_order,rollout_speed_qoq_change,first_dose_progress_qoq_change,full_vaccination_progress_qoq_change
0,Africa,2020Q4,0.000000,0.005640,0.042366,region,1,NaN,NaN,NaN
1,Africa,2021Q1,0.004613,0.008338,0.043920,region,1,0.004613,0.002698,0.001553
2,Africa,2021Q2,0.020179,0.039624,0.059176,region,1,0.015566,0.031286,0.015256
3,Africa,2021Q3,0.044275,0.109457,0.114269,region,1,0.024097,0.069833,0.055093
4,Africa,2021Q4,0.077383,0.258464,0.240665,region,1,0.033108,0.149008,0.126396
5,Africa,2022Q1,0.065326,0.434440,0.388338,region,1,-0.012057,0.175976,0.147673
6,Africa,2022Q2,0.042637,0.553382,0.507059,region,1,-0.022688,0.118942,0.118721
7,Africa,2022Q3,0.055789,0.685719,0.651389,region,1,0.013152,0.132337,0.144330
8,Africa,2022Q4,0.055936,0.816398,0.795267,region,1,0.000146,0.130678,0.143878
9,Africa,2023Q1,0.025177,0.900650,0.896095,region,1,-0.030759,0.084252,0.100828


In [29]:
peak_change_rows = []

for country, temp in group_qoq_change.groupby("country"):
    for metric in ["rollout_speed", "first_dose_progress", "full_vaccination_progress"]:
        col = f"{metric}_qoq_change"
        temp2 = temp.dropna(subset=[col]).sort_values(col, ascending=False)
        
        if len(temp2) == 0:
            continue
        
        best = temp2.iloc[0]
        
        peak_change_rows.append({
            "country": country,
            "metric": metric,
            "peak_change_quarter": best["quarter"],
            "peak_qoq_change": best[col]
        })

group_peak_change_summary = pd.DataFrame(peak_change_rows)
group_peak_change_summary.head(20)

,country,metric,peak_change_quarter,peak_qoq_change
0,Africa,rollout_speed,2021Q4,0.033108
1,Africa,first_dose_progress,2022Q1,0.175976
2,Africa,full_vaccination_progress,2022Q1,0.147673
3,Asia,rollout_speed,2021Q3,0.169536
4,Asia,first_dose_progress,2021Q4,0.264024
5,Asia,full_vaccination_progress,2021Q4,0.264878
6,Europe,rollout_speed,2021Q2,0.183384
7,Europe,first_dose_progress,2021Q3,0.341560
8,Europe,full_vaccination_progress,2021Q3,0.426194
9,European Union (27),rollout_speed,2021Q2,0.302464


In [30]:
group_qoq_change.to_csv("group_qoq_change.csv", index=False)
group_peak_change_summary.to_csv("group_peak_change_summary.csv", index=False)

In [31]:
milestone_thresholds = [0.25, 0.50, 0.75]

milestone_rows = []

for country, temp in group_quarterly.groupby("country"):
    temp = temp.sort_values("quarter")
    
    for metric in ["first_dose_progress", "full_vaccination_progress"]:
        for threshold in milestone_thresholds:
            reached = temp[temp[metric] >= threshold]
            
            milestone_rows.append({
                "country": country,
                "metric": metric,
                "threshold": threshold,
                "first_quarter_reached": reached.iloc[0]["quarter"] if len(reached) > 0 else np.nan
            })

group_milestone_table = pd.DataFrame(milestone_rows)
group_milestone_table.head(20)

,country,metric,threshold,first_quarter_reached
0,Africa,first_dose_progress,0.25,2021Q4
1,Africa,first_dose_progress,0.50,2022Q2
2,Africa,first_dose_progress,0.75,2022Q4
3,Africa,full_vaccination_progress,0.25,2022Q1
4,Africa,full_vaccination_progress,0.50,2022Q2
5,Africa,full_vaccination_progress,0.75,2022Q4
6,Asia,first_dose_progress,0.25,2021Q3
7,Asia,first_dose_progress,0.50,2021Q3
8,Asia,first_dose_progress,0.75,2021Q4
9,Asia,full_vaccination_progress,0.25,2021Q2


In [32]:
group_milestone_table.to_csv("group_milestone_table.csv", index=False)

In [33]:
group_overview_cards = (
    group_quarterly.groupby("country")
    .agg(
        latest_quarter=("quarter", "max"),
        peak_rollout_speed=("rollout_speed", "max"),
        avg_rollout_speed=("rollout_speed", "mean"),
        max_first_dose_progress=("first_dose_progress", "max"),
        max_full_vaccination_progress=("full_vaccination_progress", "max")
    )
    .reset_index()
)

group_overview_cards = group_overview_cards.merge(group_meta, on="country", how="left")

for col in ["peak_rollout_speed", "avg_rollout_speed", "max_first_dose_progress", "max_full_vaccination_progress"]:
    group_overview_cards[col] = group_overview_cards[col].round(4)

group_overview_cards

,country,latest_quarter,peak_rollout_speed,avg_rollout_speed,max_first_dose_progress,max_full_vaccination_progress,group_type,display_order
0,Africa,2024Q3,0.0774,0.0263,1.0,1.0,region,1
1,Asia,2024Q3,0.2798,0.0443,1.0,1.0,region,2
2,Europe,2024Q3,0.3141,0.0479,1.0,1.0,region,3
3,European Union (27),2024Q3,0.4250,0.0516,1.0,1.0,region_special,7
4,High-income countries,2024Q3,0.3001,0.0530,1.0,1.0,income,8
5,Low-income countries,2024Q3,0.0632,0.0220,1.0,1.0,income,11
6,Lower-middle-income countries,2024Q3,0.2070,0.0451,1.0,1.0,income,10
7,North America,2024Q3,0.2517,0.0535,1.0,1.0,region,4
8,Oceania,2024Q3,0.3317,0.0443,1.0,1.0,region,6
9,South America,2024Q3,0.3592,0.0596,1.0,1.0,region,5


In [34]:
group_overview_cards.to_csv("group_overview_cards.csv", index=False)

In [35]:
region_quarterly = group_quarterly[group_quarterly["group_type"].isin(["region", "region_special", "global"])].copy()
income_quarterly = group_quarterly[group_quarterly["group_type"] == "income"].copy()

region_quarterly.to_csv("region_quarterly_rollout.csv", index=False)
income_quarterly.to_csv("income_quarterly_rollout.csv", index=False)

In [36]:
group_summary = (
    group_quarterly.groupby("country")
    .agg(
        peak_rollout_speed=("rollout_speed", "max"),
        avg_rollout_speed=("rollout_speed", "mean"),
        final_first_dose_progress=("first_dose_progress", "max"),
        final_full_vaccination_progress=("full_vaccination_progress", "max")
    )
    .reset_index()
)

peak_quarter_rows = []

for country, temp in group_quarterly.groupby("country"):
    temp = temp.sort_values("rollout_speed", ascending=False)
    best = temp.iloc[0]

    peak_quarter_rows.append({
        "country": country,
        "peak_rollout_quarter": best["quarter"],
        "peak_rollout_speed": best["rollout_speed"]
    })

peak_quarter_df = pd.DataFrame(peak_quarter_rows)

group_summary = group_summary.merge(peak_quarter_df, on=["country", "peak_rollout_speed"], how="left")
group_summary = group_summary.merge(group_meta, on="country", how="left")

group_summary.to_csv("group_summary.csv", index=False)